In [ ]:
"""
Automated Protein Extraction (APE)

Extract and purify HIS-tagged proteins from E. coli lysates.

Columns used in the deepwell plate:
    # col 1-- Lysate samples in deepwell plate (500 µL supernatant)
    # col 2-- 3x binding buffer + mag beads (1000 µL)
    # col 3-- Wash buffer 1 (1000 µL)1]
    # col 4-- Wash buffer 2 (1000 µL)
    # col 5-- Wash buffer 3 (1000 µL)
    # col 6-- Empty column to receive eluted proteins (100 µL elution buffer)
r
# Don't forget to set a beginning column for the 1000ul tips below.
e.g. BEGIN_COLUMN = 10
# Don't forget the 96w alpaqua plate on rail 7; must be on 3d printed (10mm) supports

Author : Harley King
Date   : 2025-10-30
Update: 2025-11-14 successfully tested on Hamilton STARlet
Update: 2026-02-22 updated CORE gripper movement
"""


"\nAutomated Protein Extraction (APE)\n\nExtract and purify HIS-tagged proteins from E. coli lysates.\n\nColumns used in the deepwell plate:\n    # col 1-- Lysate samples in deepwell plate (500 µL supernatant)\n    # col 2-- 3x binding buffer + mag beads (1000 µL)\n    # col 3-- Wash buffer 1 (1000 µL)1]\n    # col 4-- Wash buffer 2 (1000 µL)\n    # col 5-- Wash buffer 3 (1000 µL)\n    # col 6-- Empty column to receive eluted proteins (100 µL elution buffer)\nr\n# Don't forget to set a beginning column for the 1000ul tips below.\ne.g. BEGIN_COLUMN = 10\n# Don't forget the 96w alpaqua plate on rail 7; must be on 3d printed (10mm) supports\n\nAuthor : Harley King\nDate   : 2025-10-30\nUpdate: 2025-11-14 successfully tested on Hamilton STARlet\n"

In [2]:


# --- Notebook conveniences (safe to ignore when running as a script) ---
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass


import asyncio
from typing import List
import time
import random

In [3]:



from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import (
    Hamilton_MFX_plateholder_DWP_metal_tapped,
)
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.alpaqua import Alpaqua_96_magnum_flx
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL, # 1000 µL filtered
    hamilton_96_tiprack_10uL_filter, # 10 µL filtered
)
from pylabrobot.liquid_handling.standard import Mix
import time

# --------------------------------------------------------------------------------------
# Deck & labware setup (matches positions/plasticware described in the user snippet)
# --------------------------------------------------------------------------------------
backend = STARBackend()
lh: LiquidHandler = LiquidHandler(backend=backend, deck=STARLetDeck())

deck = STARLetDeck(
  core_grippers="1000uL-at-waste"  # or "1000uL-5mL-on-waste"
) 

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

# No computer or cables are included - unit is as pictured.
tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10


# Magnetic plate on rails=13, module slot 0
# dwp_mod_mag = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_mag")
# car_13 = MFX_CAR_L5_base("car_13", modules={0: dwp_mod_mag})
# lh.deck.assign_child_resource(car_13, rails=13)
# mag_plate = Alpaqua_96_magnum_flx("mag_plate")
# dwp_mod_mag.assign_child_resource(mag_plate)

# Magnetic plate on rails=7, module slot 0
dwp_mod_mag = Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("dwp_mod_mag")
car_7 = MFX_CAR_L5_base("car_7", modules={0: dwp_mod_mag})
lh.deck.assign_child_resource(car_7, rails=7)
mag_plate = Alpaqua_96_magnum_flx("mag_plate")
dwp_mod_mag.assign_child_resource(mag_plate)


# Deep-well plate with samples and reagents on rails=19, module slot 0
dwp_mod_dw = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dw")
car_19 = MFX_CAR_L5_base("car_19", modules={0: dwp_mod_dw})
lh.deck.assign_child_resource(car_19, rails=19)


dw_plate = BioER_96_wellplate_Vb_2200uL("dw_plate")
dwp_mod_dw.assign_child_resource(dw_plate)

await lh.setup(skip_autoload=True)

# STARlet without iSWAP reports 0 arms; but we still want Co-Re gripper moves.
if lh.backend.num_arms == 0:
    lh._resource_pickups = {0: None}  # pragmatic workaround
print(lh.deck.get_resource("core_grippers"))

/tmp/ipykernel_902265/3306911381.py:49: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_7 = MFX_CAR_L5_base("car_7", modules={0: dwp_mod_mag})
/tmp/ipykernel_902265/3306911381.py:51: DeprecationWarning: Alpaqua_96_magnum_flx is deprecated. Use 'alpaqua_96_plateadapter_magnum_flx' instead.
  mag_plate = Alpaqua_96_magnum_flx("mag_plate")
/tmp/ipykernel_902265/3306911381.py:56: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  dwp_mod_dw = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dw")
/tmp/ipykernel_902265/3306911381.py:57: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_19 = MFX_CAR_L5_base("car_19", modules={0: dwp_mod_dw})
2026-02-20 18:28:57,067 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-20 18:28:57,076 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-20 18:28:57,

HamiltonCoreGrippers(name='core_grippers', location=Coordinate(022.500, -29.500, 105.000), size_x=39, size_y=61, size_z=24, category=core_grippers)


In [ ]:
# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------
ROWS = ["A","B","C","D","E","F","G","H"]
CHANNELS = list(range(8))  # 8-channel pipetting

async def m_mix(plateCol, mixvol=900): # wells: List, volume: int, repetitions: int = 5):
    well_A1 = dw_plate.get_item("A1")
    lheight = well_A1.compute_height_from_volume(mixvol)
    await lh.aspirate(
            plateCol, 
            vols=[mixvol]*8,
            use_channels=CHANNELS,
            liquid_height = [lheight-1]*8,
            auto_surface_following_distance=True,
            flow_rates=[400]*8,
        )
    await lh.dispense(
            plateCol,
            vols=[mixvol]*8,
            use_channels=CHANNELS,
            liquid_height = [6]*8, # the perfect height for 1000ul. About 1mm when finished. 
            flow_rates=[400]*8,
            mix=[Mix(volume=mixvol, repetitions=2, flow_rate=400)]*8,           
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
async def remove_supernatant(col_from, col_to, vol=1000, final_dispense_height=22):
    plate = dw_plate["A1"]
    # Decide pass plan (your rule: 2 passes if > 1000; split evenly)
    if vol > 1000:
        passes = 2
        pass_vols = [vol / 2, vol - (vol / 2)]  # e.g., 1500 -> [750, 750]
    else:
        passes = 1
        pass_vols = [vol]

    remaining = vol
    for i in range(passes):
        this_pass = pass_vols[i]
        # Compute lheight from the *current* remaining volume
        # e.g., vol=1500: pass 1 uses height(1500) ≈ 25; pass 2 uses height(750)
        well_A1 = dw_plate.get_item("A1")
        lheight = well_A1.compute_height_from_volume(remaining)
        # Aspirate
        await lh.aspirate(
            col_from,
            vols=[this_pass]*8,
            use_channels=CHANNELS,
            liquid_height=[lheight-1]*8, # make sure it makes contact with liquid
            auto_surface_following_distance=True,
        )
        # Update remaining volume for next pass
        remaining -= this_pass
        if remaining < 0:
            remaining = 0
        # Dispense
        await lh.dispense(
            col_to,
            vols=[this_pass]*8,
            use_channels=CHANNELS,
            liquid_height=[final_dispense_height]*8,   # your chosen target height in the destination
            blow_out=[1]*8,
            settling_time=[1]*8
        )
    # final droplets; suck it dry
    await lh.aspirate(
            col_from, 
            vols=[200]*8,
            use_channels=CHANNELS,
            liquid_height = [0]*8, # mag plate spring loaded
        )
    await lh.dispense(
            col_to,
            vols=[200]*8,
            use_channels=CHANNELS,
            liquid_height = [final_dispense_height]*8,     
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    

async def move_dw_to_mag(gripOne: int, gripTwo: int):
    mag_plate.plate_z_offset = -5.5  # mm

    # STARlet w/out iSWAP: allow PLR resource pickup bookkeeping
    if lh.backend.num_arms == 0:
        lh._resource_pickups = {0: None}

    # core_front_channel is 0-indexed and must be adjacent pair
    if abs(gripOne - gripTwo) != 1:
        raise ValueError("Co-Re gripper channels must be adjacent (e.g., 6 & 7).")
    core_front_channel = max(gripOne, gripTwo)
    if core_front_channel == 0:
        raise ValueError("core_front_channel cannot be 0 (front channel must be >= 1).")

    # ---- MOVE: keep tools mounted so the subsequent push-flush can run ----
    await lh.move_plate(
        plate=dw_plate,
        to=mag_plate,
        use_arm="core",
        core_front_channel=core_front_channel,     # new API :contentReference[oaicite:1]{index=1}
        core_grip_strength=50,
        pickup_distance_from_top=10,
        enable_recovery=True,

        return_core_gripper=False,                 # <-- critical :contentReference[oaicite:2]{index=2}
    )

    # ---- VERIFY + PUSH-FLUSH (requires grippers mounted) ----
    await lh.backend.core_check_resource_exists_at_location_center(
        location=dw_plate.get_absolute_location(),
        resource=dw_plate,
        gripper_y_margin=9,
        enable_recovery=True,
        audio_feedback=False,
    )

    # ---- PARK TOOLS ----
    await lh.backend.return_core_gripper_tools()   # :contentReference[oaicite:3]{index=3}



async def move_dw_back(gripOne: int, gripTwo: int):
    try:
        # ---- STARlet w/out iSWAP: allow PLR resource pickup bookkeeping ----
        if lh.backend.num_arms == 0:
            lh._resource_pickups = {0: None}

        # ---- new API: core_front_channel (0-indexed) ----
        if abs(gripOne - gripTwo) != 1:
            raise ValueError("Co-Re gripper channels must be adjacent (e.g., 6 & 7).")
        core_front_channel = max(gripOne, gripTwo)  # front channel cannot be 0
        if core_front_channel == 0:
            raise ValueError("core_front_channel cannot be 0 (front channel must be >= 1).")

        # ---- MOVE: keep tools mounted if you want push-flush / verification ----
        await lh.move_plate(
            plate=dw_plate,
            to=dwp_mod_dw,
            use_arm="core",
            pickup_distance_from_top=10,
            core_front_channel=core_front_channel,
            core_grip_strength=50,
            enable_recovery=False,

            return_core_gripper=False,  # keep tools mounted so core_check can run
        )
        # recovery not needed for plate as this is hard on grippers
        # # ---- VERIFY + PUSH-FLUSH (optional but recommended) ----
        # await lh.backend.core_check_resource_exists_at_location_center(
        #     location=dw_plate.get_absolute_location(),
        #     resource=dw_plate,
        #     gripper_y_margin=9,
        #     enable_recovery=True,
        #     audio_feedback=False,
        # )

        # ---- PARK TOOLS ----
        await lh.backend.return_core_gripper_tools()

    except Exception as e:
        print("[WARN] Implement gripper move for automated transfer back.")
        print("Exception:", e)



In [ ]:
# --------------------------------------------------------------------------------------
# Main functions
# --------------------------------------------------------------------------------------

async def add_binding_buffer_and_mix(tipColumn =1):
    """Function 1: Add 3xBB-b, mix and incubate; capture on magnet, remove sup."""
    col1 = dw_plate["A1:H1"]
    col2 = dw_plate["A2:H2"]

    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    try:
        await lh.aspirate(
            col2, 
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height = [2]*8,
            mix=[Mix(volume=500, repetitions=3, flow_rate=400)]*8, # mix settled beads
        )
        await lh.dispense(
            col1,
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height = [20]*8, # the perfect height for 1000ul. About 1mm when finished.            
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
        # get any last remaining beads
        await lh.aspirate(
            col2, 
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height = [0]*8,
            flow_rates=[100]*8,
        )
        await lh.dispense(
            col1,
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height = [20]*8, # the perfect height for 1000ul. About 1mm when finished.            
            flow_rates=[100]*8,
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    except Exception as e:
        print("[WARN] Implement error handling for aspirate/dispense 3xBB-b.")
        print (e)
    # calculating mixing/incubation time
    premixing = time.time()
    for n in range(10):
        await m_mix(col1, mixvol=1000)  
        time.sleep(2*60) # incubate 4 minutes
        print ("This is round ", n, " of 10 mixing/incubation steps.")
    postmixing = time.time()
    print(f"Total mixing and incubation time: {round((postmixing - premixing)/60,2)} minutes")
    await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    gripOne = random.randint(1, 6) 
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    time.sleep(2)  # Incubate 2 minutes
    # protein bound to beads. Remove unbound supernatant to col2
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await remove_supernatant(col1, col2, 1500)
    await lh.discard_tips() # tips very messy
    await move_dw_back(gripOne, gripTwo)
    print ("Binding step complete in col 1. Ready for washes.")

async def wash_step(tipColumn=1, wash_col=3):
    """Function 2 & 3: Wash with wash buffer 1 and 2; capture on magnet, remove sup."""
    col1 = dw_plate["A1:H1"]
    wash_col = dw_plate[f"A{wash_col}:H{wash_col}"]

    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await lh.aspirate(
            wash_col, # add 1500ul
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height = [20]*8,
            auto_surface_following_distance=True,
        )
    await lh.dispense(
            col1,
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height = [10]*8, # the perfect height for 1000ul. About 1mm when finished. 
            flow_rates=[400]*8,
            mix=[Mix(volume=1000, repetitions=2, flow_rate=400)]*8,           
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    for n in range(3):
        await m_mix(col1, mixvol=1000)
        time.sleep(1*60)  
    await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    gripOne = random.randint(1, 6) 
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    time.sleep(1*60)  # Incubate 2 minutes
    # protein bound to beads. Remove wash supernatant to col2
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await remove_supernatant(col1, wash_col, 1000) # remove contents of col1 into wash_col
    await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS) # tip can be reused
    await move_dw_back(gripOne, gripTwo)
    print (f"Wash step complete. Wash buffer in col{wash_col}.")

async def final_wash_and_bead_transfer(tipColumn=1, final_wash_buffer_col=5):
    """Complete final wash and transfer beads to fresh well to minimize carryover of proteins.
    At the end of the wash_step function above, the beads should be in col1 congregating at the bottom(4mm).
    The plate is off the magnet."""
    col1 = dw_plate["A1:H1"]
    dest_col = dw_plate[f"A{final_wash_buffer_col}:H{final_wash_buffer_col}"]
    
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await lh.aspirate(
            dest_col, 
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height = [1]*8, 
        )
    await lh.dispense(
            col1,
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height = [2]*8,     
            mix=[Mix(volume=500, repetitions=4, flow_rate=300)]*8, # slow flow so beads don't splash
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    # await m_mix(col1, mixvol=500) # in testing, some beads remain stuck at bottom
    # aspirate beads and transfer to dest_col
    await lh.aspirate(
            col1, 
            vols=[500]*8,
            use_channels=CHANNELS,
            mix=[Mix(volume=500, repetitions=2, flow_rate=200)]*8, # slow flow so beads don't splash
            liquid_height = [1]*8, 
        )
    await lh.dispense(
            dest_col,
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height = [10]*8,     
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    # beads now in dest_col. Wash by mixing
    for n in range(3):
        await m_mix(dest_col, mixvol=900)
        time.sleep(1*60)  #a little incubation time
    await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    
    gripOne = random.randint(1, 6) 
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    time.sleep(1*60)  # Incubate 2 minutes
    # beads on magnet. Transfer beads to dest_col
    # beads collect in col4. Remove supernatant to col1
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await remove_supernatant(dest_col, col1, 1000) # remove supernatant from dest_col (4) to col1
    # await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS) # tips can be reused
    await lh.discard_tips()
    await move_dw_back(gripOne, gripTwo)
    print ("Final wash complete. Beads transferred to fresh well.")

async def elution_step(tipColumn=1, beads_in_col =5, elute_in_col=6):
    """Elute proteins with elution buffer; 
    capture beads on magnet, transfer eluted proteins.
    Would love to heat at 60C on heating shaking platform"""
    # col4 = dw_plate["A4:H4"]
    beadCol = dw_plate[f"A{beads_in_col}:H{beads_in_col}"]
    eluteCol = dw_plate[f"A{elute_in_col}:H{elute_in_col}"]
    # think about using 50ul tips for fewer carry-over beads?
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await lh.aspirate(
            eluteCol, 
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height = [1]*8,            
        )
    await lh.dispense(
            beadCol,
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height = [1]*8,
            mix=[Mix(volume=100, repetitions=6, flow_rate=50)]*8, 
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
    time.sleep(2*60)  #a little incubation time
    await m_mix(beadCol, mixvol=100)
    await lh.drop_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    gripOne = random.randint(1, 6) 
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    # eluted protein in supernatant. Transfer to col6
    await lh.pick_up_tips(tiprack_1000[f"A{tipColumn}:H{tipColumn}"], use_channels=CHANNELS)
    await remove_supernatant(beadCol, eluteCol, 100, final_dispense_height=5)
    await lh.discard_tips()
    await move_dw_back(gripOne, gripTwo)
    print ("Elution complete. Eluted")
    # protein in col6 without beads




In [8]:


# --------------------------------------------------------------------------------------
# Column mapping
# --------------------------------------------------------------------------------------
# col 1-- Lysate samples in deepwell plate (500 µL supernatant)
# col 2-- 3x binding buffer + mag beads (1000 µL)
# col 3-- Wash buffer 1 (1000 µL)1]
# col 4-- Wash buffer 2 (1000 µL)
# col 5-- Wash buffer 3 (1000 µL)
# col 6-- Empty column to receive eluted proteins (100 µL elution buffer)
# await lh.setup()


BEGIN_COLUMN =6

await add_binding_buffer_and_mix(tipColumn=BEGIN_COLUMN)
await wash_step(tipColumn=BEGIN_COLUMN+1, wash_col=3)
await wash_step(tipColumn=BEGIN_COLUMN+1, wash_col=4) #same tips can be reused for wash steps
await final_wash_and_bead_transfer(tipColumn=BEGIN_COLUMN+1, final_wash_buffer_col=5)
await elution_step(tipColumn=BEGIN_COLUMN+2, beads_in_col =5, elute_in_col=6)


This is round  0  of 10 mixing/incubation steps.
This is round  1  of 10 mixing/incubation steps.
This is round  2  of 10 mixing/incubation steps.
This is round  3  of 10 mixing/incubation steps.
This is round  4  of 10 mixing/incubation steps.
This is round  5  of 10 mixing/incubation steps.
This is round  6  of 10 mixing/incubation steps.
This is round  7  of 10 mixing/incubation steps.
This is round  8  of 10 mixing/incubation steps.
This is round  9  of 10 mixing/incubation steps.
Total mixing and incubation time: 25.5 minutes
Binding step complete in col 1. Ready for washes.
Wash step complete. Wash buffer in col[Well(name='dw_plate_well_A3', location=Coordinate(027.500, 070.500, 006.000), size_x=8.25, size_y=8.25, size_z=42.4, category=well), Well(name='dw_plate_well_B3', location=Coordinate(027.500, 061.500, 006.000), size_x=8.25, size_y=8.25, size_z=42.4, category=well), Well(name='dw_plate_well_C3', location=Coordinate(027.500, 052.500, 006.000), size_x=8.25, size_y=8.25, size

In [ ]:

# col 1-- Lysate samples in deepwell plate (500ul supernatant)
# col 2-- 3x binding buffer + mag beads (1100ul)
# col 3-- Wash buffer 1 (1500ul)
# col 4-- Wash buffer 2 (1500ul)
# col 5-- Elution buffer (100ul)
# col 6-- Empty column to receive eluted proteins (100ul)
# await lh.backend.return_core_gripper_tools() 

# Function 1: Add 3xBB-b, mix and incubate
#1.1 Mix col 2 containing 1100ul 3x binding buffer with mag beads (3xBB-b) with 8 channels
#1.2 Transfer 1000ul 3xBB-b from col 2 to col 1 containing lysate. 
#1.3 Mix col 1 containing lysate + binding buffer + mag beads. Pause for N minutes.
#1.4 Repeat step 1.3 O times
#1.5 Move dw plate to mag module, pause N minutes, remove supernatant to col 2. 
#1.6 Remove dw plate from mag module and return to original position.

# Function 2: Wash beads 3x
#2.1 Add 1500ul wash buffer from col 3 to col 1. 
#2.2 Mix, pause P minutes; repeat Q times
#2.3 Move dw plate to mag module, pause N minutes, Remove supernatant to col 3
#2.4 Remove dw plate from mag module and return to original position. 
#2.5 Repeat 2.1 to 2.4 for wash buffer in col 4.

# Function 3: Elute proteins
#3.1 Add 100ul elution buffer from col 5 to col 1
#3.2 Mix, pause R minutes; repeat S times
#3.3 Move dw plate to mag module, pause N minutes, transfer eluted proteins from col 1 to col 6.
#3.4 Remove dw plate from mag module and return to original position.
#3.5 End of protocol.

# async def add_binding_buffer():



In [ ]:
# await dispense_mastermix()
# await add_template()

In [ ]:
# await lh.dispense(dw_plate["A1:H1"], vols=[100]*8, liquid_height=[20]*8, use_channels=CHANNELS)
# await lh.drop_tips(tiprack_1000["C7:H7"], use_channels=[2,3,4,5,6,7])
# await lh.drop_tips(tiprack_1000["A6:H6"], use_channels=CHANNELS)
# await lh.drop_tips(tiprack_1000["F6:H6"], use_channels=[5,6,7])
# await lh.discard_tips()
# # await lh.stop()
# await backend.return_core_gripper_tools()